In [31]:
import json
from ragas.testset.graph import KnowledgeGraph, Node, NodeType

# 1. JSON 데이터 로드 (변수명 items로 가정)
# items = json.loads(your_json_string)

kg = KnowledgeGraph()

with open('/data/ephemeral/pro-nlp-finalproject-nlp-13/data/guidebook/busan_rag_data.json', 'r', encoding='utf-8') as f:
    json_data = json.load(f)

json_data = json_data[:5]

for item in json_data:
    # 2. 본문 합치기 (설명뿐만 아니라 영업시간 등 유용한 정보 통합)
    # Ragas가 읽고 질문을 만들 "교과서" 본문을 만듭니다.
    page_content = f"장소: {item['장소명']}\n설명: {item['설명']}\n영업시간: {item['영업시간']}\n연락처: {item['연락처']}"

    # 3. 노드 생성
    node = Node(
        type=NodeType.CHUNK, # MySingleHopScenario에서 CHUNK를 찾으므로 그대로 유지
        properties={
            "page_content": page_content,
            # 시나리오 생성기가 핵심 주제로 인식할 키워드들
            "keyphrases": [item['장소명'], item['구분']], 
            "document_metadata": {
                "source": item['출처'],
                "contact": item['연락처']
            }
        }
    )
    
    kg.nodes.append(node)

print(f"✅ {len(kg.nodes)}개의 장소 정보가 그래프로 변환되었습니다.")

✅ 5개의 장소 정보가 그래프로 변환되었습니다.


In [32]:
from openai import OpenAI
from ragas.llms import llm_factory
from ragas.embeddings import OpenAIEmbeddings
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

openai_client = OpenAI(api_key=api_key)
llm = llm_factory("gpt-4o-mini", client=openai_client)
embedding = OpenAIEmbeddings(client=openai_client)

In [33]:
from ragas.testset.transforms import apply_transforms
from ragas.testset.transforms import (
    HeadlinesExtractor,
    HeadlineSplitter,
    KeyphrasesExtractor,
)


headline_extractor = HeadlinesExtractor(llm=llm)
headline_splitter = HeadlineSplitter(min_tokens=30, max_tokens=100)
keyphrase_extractor = KeyphrasesExtractor(
    llm=llm, property_name="keyphrases", max_num=10
)

transforms = [
    headline_extractor,
    headline_splitter,
    keyphrase_extractor,
]

apply_transforms(kg, transforms=transforms)

Applying HeadlinesExtractor:   0%|          | 0/5 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/5 [00:00<?, ?it/s]

Applying KeyphrasesExtractor:   0%|          | 0/20 [00:00<?, ?it/s]

Property 'keyphrases' already exists in node '8f818e'. Skipping!
Property 'keyphrases' already exists in node '47faec'. Skipping!
Property 'keyphrases' already exists in node 'f43a5c'. Skipping!
Property 'keyphrases' already exists in node '8de6ef'. Skipping!
Property 'keyphrases' already exists in node '9c007b'. Skipping!


In [34]:
from ragas.testset.persona import Persona

# 부산 여행 가이드나 축제 데이터에 어울리는 페르소나 예시
traveler = Persona(
    name="First-time visitor to Busan",
    role_description="부산에 처음 놀러 온 20대 여행객. 사진 찍기 좋은 곳과 돼지갈비집을 궁금해함.",
)
persona_list = [traveler]

In [35]:
from ragas.testset.synthesizers.single_hop import (
    SingleHopQuerySynthesizer,
    SingleHopScenario,
)
from dataclasses import dataclass
from ragas.testset.synthesizers.prompts import (
    ThemesPersonasInput,
    ThemesPersonasMatchingPrompt,
)


@dataclass
class MySingleHopScenario(SingleHopQuerySynthesizer):

    theme_persona_matching_prompt = ThemesPersonasMatchingPrompt()

    async def _generate_scenarios(self, n, knowledge_graph, persona_list, callbacks):

        property_name = "keyphrases"
        nodes = []
        for node in knowledge_graph.nodes:
            if node.type.name == "CHUNK" and node.get_property(property_name):
                nodes.append(node)

        number_of_samples_per_node = max(1, n // len(nodes))

        scenarios = []
        for node in nodes:
            if len(scenarios) >= n:
                break
            themes = node.properties.get(property_name, [""])
            prompt_input = ThemesPersonasInput(themes=themes, personas=persona_list)
            persona_concepts = await self.theme_persona_matching_prompt.generate(
                data=prompt_input, llm=self.llm, callbacks=callbacks
            )
            base_scenarios = self.prepare_combinations(
                node,
                themes,
                personas=persona_list,
                persona_concepts=persona_concepts.mapping,
            )
            scenarios.extend(
                self.sample_combinations(base_scenarios, number_of_samples_per_node)
            )

        return scenarios

query = MySingleHopScenario(llm=llm)

scenarios = await query.generate_scenarios(
    n=5, knowledge_graph=kg, persona_list=persona_list
)

scenarios[0]

SingleHopScenario(
nodes=1
term=168더데크
persona=name='First-time visitor to Busan' role_description='부산에 처음 놀러 온 20대 여행객. 사진 찍기 좋은 곳과 돼지갈비집을 궁금해함.'
style=Poor grammar
length=short)

In [36]:
result = await query.generate_sample(scenario=scenarios[-1])

result

SingleTurnSample(user_input="부산에서 30년 전통의 돼지갈비 전문점인 '가마솥석쇠돼지갈비'에 대해 자세히 알고 싶습니다. 이 음식점의 특징과 추천 메뉴는 무엇인가요?", retrieved_contexts=None, reference_contexts=['장소: 30년전통가마솥석쇠돼지갈비\n설명: 국내산 돼지를 사용하여 초벌구이된 양념돼지갈비 단일메뉴로 30여 년을 운영해온 돼지갈비전문점으로 모든 음식을 직접 만든다. 식사를 주문하면 제공되는 찌개와 게조림이 기대 이상의 맛을 보여준다.\n영업시간: 12:00-22:00, 연중 무휴\n연락처: 0507-1305-6175'], retrieved_context_ids=None, reference_context_ids=None, response=None, multi_responses=None, reference='가마솥석쇠돼지갈비는 국내산 돼지를 사용하여 초벌구이된 양념돼지갈비 단일 메뉴로 30여 년을 운영해온 돼지갈비 전문점입니다. 모든 음식은 직접 만들어 제공되며, 식사를 주문하면 찌개와 게조림이 함께 제공되어 기대 이상의 맛을 경험할 수 있습니다. 영업시간은 12:00부터 22:00까지이며, 연중 무휴로 운영됩니다. 연락처는 0507-1305-6175입니다.', rubrics=None, persona_name='First-time visitor to Busan', query_style='WEB_SEARCH_LIKE', query_length='LONG')

In [37]:
instruction = """제공된 컨텍스트와 조건(페르소나, 용어, 스타일, 길이)을 바탕으로 자연스러운 질의응답 세트를 생성하세요. 
모든 답변은 반드시 컨텍스트의 정보에만 철저히 근거해야 합니다.

### 상세 지침:
1. **질문 생성 (Query)**: 
   - '페르소나'의 역할과 상황에 몰입하여 질문을 작성하세요.
   - 컨텍스트 속의 '용어(term)'를 자연스럽게 포함해야 합니다.
   - '예/아니오' 단답형 질문이 아닌, 구체적인 정보나 설명을 요구하는 질문을 생성하세요.

2. **답변 생성 (Answer)**: 
   - 컨텍스트의 내용을 활용하여 질문에 대해 상세하고 친절하게 답변하세요.
   - **절대로 외부 지식을 사용하지 마세요.** 컨텍스트에 없는 내용은 "정보가 부족하여 답변할 수 없다"고 명시해야 합니다.
   - 페르소나가 기대하는 정보의 핵심을 정확히 짚어주세요.

3. **언어 및 스타일**: 
   - 모든 입출력은 한국어로 작성합니다.
   - 지정된 '스타일'과 '길이' 조건을 엄격히 준수하세요.
"""


In [38]:
print("사용 가능한 프롬프트 키:", query.get_prompts().keys())

사용 가능한 프롬프트 키: dict_keys(['query_answer_generation_prompt', 'themes_personas_matching_prompt'])


In [39]:
prompt = query.get_prompts()["query_answer_generation_prompt"]
prompt.instruction = instruction
query.set_prompts(**{"query_answer_generation_prompt": prompt})
result = await query.generate_sample(scenario=scenarios[-1])

In [43]:
result.user_input

'부산에서 30년 전통의 가마솥석쇠돼지갈비 음식점에 대해 자세히 알려주세요. 특히 어떤 메뉴가 추천되는지와 영업시간, 연락처도 포함해서 설명해 주시면 좋겠습니다.'

In [45]:
result.reference_contexts

['장소: 30년전통가마솥석쇠돼지갈비\n설명: 국내산 돼지를 사용하여 초벌구이된 양념돼지갈비 단일메뉴로 30여 년을 운영해온 돼지갈비전문점으로 모든 음식을 직접 만든다. 식사를 주문하면 제공되는 찌개와 게조림이 기대 이상의 맛을 보여준다.\n영업시간: 12:00-22:00, 연중 무휴\n연락처: 0507-1305-6175']

In [46]:
result.retrieved_contexts

In [49]:
result.reference

'30년 전통가마솥석쇠돼지갈비는 국내산 돼지를 사용하여 초벌구이된 양념돼지갈비 단일메뉴로 운영되는 돼지갈비 전문점입니다. 모든 음식은 직접 만들어 제공되며, 식사를 주문하면 찌개와 게조림이 함께 제공되어 기대 이상의 맛을 경험할 수 있습니다. 영업시간은 매일 12:00부터 22:00까지이며, 연중 무휴로 운영됩니다. 연락처는 0507-1305-6175입니다.'